# The Stability Experiment (Temperature Stress Test)

#### Crisis systems require determinism.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model, should_use_reasoning_model
from IPython.display import Markdown, display

from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()

True

In [ ]:
reasoning_model = pick_model('google','cot')
print(f'Using reasoning model: {reasoning_model}')

client_reasoning = LLMClient('google',reasoning_model)

data_path = "../data/data/Scenarios.txt"
with open(data_path, "r") as f:
    scenarios = f.read().split("\n\n")
    
print(f"Loaded {len(scenarios)} senarios")

Using reasoning model: gemini-2.5-pro
Loaded 2 senarios


In [9]:
def get_instruction():
    return """Solve the following crisis scenario step by step.
1. Identify all threats (immediate life, health, high risk, etc.).
2. Prioritize actions based on threat severity.
3. Provide specific step-by-step actions for each threat.
4. Avoid unrealistic or hallucinated suggestions.
5. Comment on any potential resource constraints and how to allocate them."""

results = {}

for idx,scenario in enumerate(scenarios):
    scenario_label = scenario.strip().split("\n")[0]
    print(f"\n{'='*80}")
    print(f"{scenario_label}")
    print(f"{'='*80}")
    
    prompt_text, spec = render(
        'cot_reasoning.v1',
        role='emergency_responder',
        problem=scenario,
    )
    
    instruction = get_instruction()
    full_prompt = f"""
    text:{prompt_text}
    instruction: {instruction}"""

    messages = [{'role':'user','content':full_prompt}]
    results[scenario_label] = {'safe':None,'chaos':[]}
    
    #Safe Mode | T = 0.0
    print(f"\n{'-'*40}")
    print("Safe Mode")
    print(f"{'-'*40}")

    safe_response = client_reasoning.chat(
        messages,temperature=0.0,max_tokens=spec.max_tokens,
        )
    results[scenario_label]['safe'] = safe_response
    display(Markdown(f"**Safe Mode Response:**\n\n{safe_response['text']}"))
    print(f"\n Latency:{safe_response['latency_ms']}ms | "
        f"Tokens:{safe_response['usage'].get('total_tokens_actual','N/A')}")

    #Chaos Mode | 3 times | T = 1.0
    print(f"\n{'-'*40}")
    print("Chaos Mode")
    print(f"{'-'*40}")

    for run in range(3):
        print(f"\n -- Run {run + 1}/3 --")
        chaos_response = client_reasoning.chat(
            messages,temperature=1.0,max_tokens=spec.max_tokens,
        )
        results[scenario_label]['chaos'].append(chaos_response)
        display(Markdown(f"**Chaos Mode response:**\n\n{chaos_response['text']}"))
        print(f"\n Latency:{chaos_response['latency_ms']}ms | "
            f"Tokens:{chaos_response['usage'].get('total_tokens_actual','N/A')}")

print(f"\n{'='*80}")
print("Experiment complete!")
print(f"{'='*80}")

    


SCENARIO A: THE KANDY LANDSLIDE

----------------------------------------
Safe Mode
----------------------------------------


**Safe Mode Response:**

**Reasoning Steps (brief)**  
1. List every threat present.  
2. Rank threats by severity (Immediate Life Threat > Immediate Health Threat > High‑Risk/Secondary threats).  
3. For each ranked threat, outline concrete actions that can be taken with the resources likely on‑site (people, basic first‑aid supplies, limited food, possible external help).  
4. Note any resource gaps and how to allocate or request additional assets.

---  

## Answer:

### 1. Identify All Threats
| Threat | Category | Why it matters |
|--------|----------|----------------|
| Uncle climbing a tree as water rises in the line rooms | **Immediate Life Threat** | Imminent drowning or fall; death possible within minutes. |
| Diabetic patient collapsed, needs insulin | **Immediate Health Threat** | Without insulin → severe hyperglycemia → coma/death in <1 hour. |
| 40 people hungry, no food for an unknown duration | **Secondary/High‑Risk Threat** | Malnutrition and dehydration become critical after several hours/days. |
| Blocked access road (landslide debris) | **Logistical/Environmental Threat** | Prevents rescue teams, supplies, and evacuation. |
| Potential further water rise / secondary landslide | **Environmental/High‑Risk Threat** | Could trap everyone or cause additional injuries. |
| Limited medical supplies, limited rescue equipment | **Resource Constraint** | Affects ability to treat and evacuate. |

### 2. Prioritize Actions
1. **Immediate Life Threat** – Uncle in rising water.  
2. **Immediate Health Threat** – Diabetic patient needing insulin.  
3. **Environmental/Logistical Threat** – Blocked road & risk of more water/landslide.  
4. **Secondary Threat** – Hunger/ dehydration of 40 people.  

### 3. Step‑by‑Step Actions  

#### A. Rescue Uncle (Immediate Life Threat)  
1. **Assemble a rescue team** – 2‑3 strong volunteers with rope or sturdy clothing.  
2. **Secure a safe anchor point** (e.g., a sturdy tree or factory beam).  
3. **Use a rope or improvised harness** to lower a rescuer to the uncle’s level.  
4. **Stabilize the uncle** – have him hold onto the rope; if possible, place a flotation device (e.g., empty oil drum, life‑jacket) under him.  
5. **Pull him up** steadily, keeping the rope taut to avoid sudden drops.  
6. **Move him to higher ground** inside the factory, away from water.  
7. **Check for injuries** (head, fractures) and provide basic first aid.  

*If rope is unavailable:*  
- Use a strong belt or shirt as a makeshift rope; tie multiple knots for strength.  
- If the water is shallow enough, have volunteers wade in with a sturdy pole to push the uncle toward safety while maintaining balance.

#### B. Treat Diabetic Patient (Immediate Health Threat)  
1. **Locate insulin** – check factory first‑aid kit, staff lockers, or any personal medication bags.  
2. **Verify dosage** – ask the patient (or a family member) for usual dose; if unknown, give a standard emergency dose (usually 1–2 U rapid‑acting insulin subcutaneously).  
3. **Administer insulin** using a sterile syringe/pen.  
4. **Check blood glucose** (if glucometer available). Aim for >150 mg/dL after injection.  
5. **Provide oral glucose** (e.g., sugary drink, glucose tablets) if the patient shows signs of hypoglycemia after insulin.  
6. **Position patient** lying down, monitor consciousness, breathing, and pulse every 5 minutes.  
7. **Call for external medical help** (ambulance, nearest hospital) as soon as possible; give exact location and condition.  

#### C. Stabilize Environment & Access (Logistical/High‑Risk)  
1. **Mark the blocked road** with visible signs to prevent further attempts.  
2. **Assign a small team** (2‑3 people) to start clearing a narrow passage using any available tools (shovels, pickaxes, even sturdy sticks). Aim for a **minimum 1‑meter wide** path for a stretcher or rescue rope.  
3. **Set up a temporary water‑drainage channel** if water continues to rise: use sandbags, tarps, or any containers to divert water away from the factory entrance.  
4. **Establish a communication point** – use a mobile phone, radio, or shouted signals to inform external rescue teams of the exact blockage location and request heavy‑equipment assistance (excavator, bulldozer).  

#### D. Address Hunger & Hydration (Secondary Threat)  
1. **Ration existing food** – divide current supplies into 3‑day emergency rations (≈ 500 kcal/person/day).  
2. **Collect water** – locate any stored water barrels, rain‑catchment, or use the factory’s boiler condensate (if safe). Boil or treat with chlorine tablets if possible.  
3. **Assign a “food‑distribution” schedule** – 2‑hourly small meals to maintain morale and prevent panic.  
4. **Identify volunteers** to monitor vulnerable individuals (elderly, children, pregnant women).  

#### E. Ongoing Safety Monitoring  
1. **Assign a “watch‑post”** near the line rooms to monitor water level every 5 minutes.  
2. **Listen for ground movement** – if any rumbling, prepare to evacuate immediately.  
3. **Maintain a simple log** of actions, resources used, and remaining supplies for incoming rescue teams.  

### 4. Resource Constraints & Allocation  

| Resource | Current Status | Allocation Plan |
|----------|----------------|-----------------|
| Rope / strong fabric | Likely limited | Use for uncle rescue; keep extra length for possible evacuation line. |
| Insulin & syringes | May be in first‑aid kit | Prioritize for diabetic patient; if insufficient, request replacement from nearest pharmacy/hospital. |
| Food (dry/ canned) | Unknown quantity | Ration for 3 days; prioritize children/elderly. |
| Water | Possible stored barrels | Boil/treat; allocate 2 L/person/day. |
| Manual tools (shovel, pickaxe) | Likely present in factory | Use for road clearing and drainage. |
| Personnel | ~40 people | Form small teams (3‑4 per task) to avoid fatigue; rotate every 30 minutes. |
| Communication device | Mobile phone likely | Use for external rescue request; conserve battery (turn off non‑essential apps). |

**If any critical resource is missing (e.g., insulin, rope, clean water):**  
- **Signal for external aid** immediately, providing exact GPS coordinates.  
- **Improvise:** use clean cloth for makeshift bandages, use sugar packets for oral glucose, use sturdy belts as rope.  

### 5. Summary of Immediate Priorities  

1. **Rescue the uncle** – life‑or‑death within minutes.  
2. **Give insulin** to the diabetic patient – prevent rapid deterioration.  
3. **Create a minimal access path** for external rescuers and for possible evacuation.  
4. **Secure water & food** for the group to survive the next 48‑72 hours.  

All actions should be communicated clearly to the group, with designated leaders for each task to maintain order and prevent panic.


 Latency:4350ms | Tokens:1921

----------------------------------------
Chaos Mode
----------------------------------------

 -- Run 1/3 --


**Chaos Mode response:**

**Reasoning Steps (brief)**  
1. List every threat present in the scene.  
2. Rank them by severity (Immediate Life Threat > Immediate Health Threat > High‑Risk/Secondary threats).  
3. Match available resources (people, first‑aid supplies, food, equipment) to the highest‑ranked threats first.  
4. Draft concrete actions for each threat, including who does what, when, and how to mitigate resource limits.  

---  

## Answer:

### 1. Identify All Threats  
| Threat | Category | Why it matters |
|--------|----------|----------------|
| Uncle climbing a tree as water rises | Immediate Life Threat | Risk of drowning, fall, or hypothermia; rescue needed now. |
| Diabetic patient collapsed, needs insulin | Immediate Health Threat | Without insulin → rapid deterioration, possible death. |
| 40 people hungry (no food/water) | Secondary Threat (Physiological stress) | Can become a health issue if not addressed, but lower immediacy. |
| Blocked access road | Logistical/High‑Risk Threat | Hinders external help and evacuation; must be cleared or bypassed. |
| Potential ongoing landslide / rising water | Environmental/High‑Risk Threat | May worsen all other threats if not monitored. |

### 2. Prioritization (Highest → Lowest)  
1. **Rescue Uncle (Immediate Life Threat).**  
2. **Treat diabetic patient (Immediate Health Threat).**  
3. **Stabilize environment – monitor water level & clear/bypass road (High‑Risk).**  
4. **Address hunger/ dehydration (Secondary).**  

### 3. Step‑by‑Step Actions  

#### A. Rescue Uncle (Immediate Life Threat)  
1. **Assemble a rescue team (3‑4 able‑bodied volunteers).**  
2. **Gather any flotation devices, ropes, or sturdy branches.**  
3. **Assign roles:**  
   - *Team Lead*: directs, maintains safety.  
   - *Rope‑handler*: secures rope around a stable anchor (tree trunk, metal beam).  
   - *Lifter*: assists Uncle onto rope/floatation.  
   - *Safety watcher*: watches water level, calls for help if situation worsens.  
4. **Execute rescue:**  
   - Throw a rope loop (lasso) to Uncle.  
   - Pull him down slowly while keeping the rope taut to avoid sudden drop.  
   - If water depth > waist, use a buoyant improvised raft (large plastic drum, wooden planks).  
5. **Move Uncle to a dry, elevated spot (≥2 m above water).**  
6. **Check for injuries, provide basic first aid (stop bleeding, keep warm).**  

#### B. Treat Diabetic Patient (Immediate Health Threat)  
1. **Locate insulin (check first‑aid kit, factory medical box).**  
2. **If insulin unavailable, request any nearby staff/visitor to retrieve it immediately.**  
3. **Check blood glucose (if glucometer present).**  
4. **If patient is unconscious or severely hypoglycemic:**  
   - Administer rapid‑acting carbohydrate (glucose tablets, sugary drink) if IV not possible.  
   - If trained personnel are present, start IV line with normal saline and give 1 U/kg rapid‑acting insulin per protocol.  
5. **Monitor vitals every 5 min until stable.**  
6. **Keep patient warm, lying supine, with legs elevated if shock suspected.**  

#### C. Stabilize Environment / Access Road  
1. **Assign two people to continuously monitor water level and any further landslide movement.**  
2. **If possible, clear a narrow passage using shovels, hand tools, or remove debris to create a 1‑meter wide lane for stretcher or evacuation.**  
3. **If road cannot be cleared quickly, identify an alternate exit (e.g., factory side door) for future evacuation.**  

#### D. Address Hunger / Dehydration (Secondary Threat)  
1. **Inventory existing food and water supplies (factory pantry, vending machines).**  
2. **Ration water: 1 L per person per day; prioritize children, elderly, and the rescued uncle.**  
3. **Distribute high‑energy, non‑perishable food (energy bars, biscuits) in small portions to keep morale and prevent hypoglycemia.**  
4. **Set up a “food station” away from the water line to avoid contamination.**  

#### E. Communication & External Help  
1. **Use any available mobile phone or radio to call local emergency services (fire, rescue, medical).**  
2. **Transmit exact location (Han​thana Tea Factory, GPS if known) and a concise status report (1 rescued, 1 diabetic patient treated, 40 people safe but need food/water).**  
3. **If signal is weak, send a runner (one strong volunteer) to the nearest road or village with a phone.**  

### 4. Resource Constraints & Allocation  

| Resource | Availability | Allocation | Mitigation if insufficient |
|----------|---------------|------------|----------------------------|
| **Rescue rope / flotation** | May be limited | Use any sturdy rope, plastic drums, wooden pallets. | Improvise with belts, belts of clothing; create a makeshift raft from pallets. |
| **Insulin** | Single vial likely | Give exact dose; share with other diabetics if any (none reported). | If insulin runs out, maintain glucose intake until EMS arrives. |
| **Medical kit** | Basic first‑aid present | Prioritize diabetic care, wound care for rescued uncle. | Use sugar sources (soft drinks) as temporary glucose. |
| **Food & water** | Factory pantry unknown | Ration immediately; prioritize water. | Collect rainwater in clean containers if safe, melt snow (if cold), or request external supply via call. |
| **Manpower** | ~40 people, varying fitness | Assign roles as above; keep a reserve team for monitoring. | Rotate teams to avoid fatigue; use the most physically able for rescue. |
| **Communication** | May have limited signal | Use one phone to call EMS; designate backup runner. | If no signal, send messenger on foot to nearest village (≈1 km). |

### 5. Immediate Summary of Actions (What to do now)

1. **Form rescue team → pull uncle to safety.**  
2. **Locate insulin → treat diabetic patient immediately.**  
3. **Assign two people to watch water/landslide, start clearing a narrow escape path.**  
4. **Call emergency services with concise status.**  
5. **Begin water/food rationing and set up distribution point.**  

By executing the steps in this order, the most lethal threats are neutralized first while maintaining a clear plan for secondary needs and external support.


 Latency:3433ms | Tokens:1774

 -- Run 2/3 --


...


 Latency:5207ms | Tokens:1875

 -- Run 2/3 --


**Chaos Mode response:**

**Reasoning Steps (brief outline)**
1. List all active threats and their severity.  
2. Rank threats by immediacy of loss of life.  
3. Match available resources (ventilator batteries, generator truck, staff, equipment) to the highest‑priority threats.  
4. Draft concrete, sequential actions for each threat, including contingency measures for resource limits.  
5. Highlight any gaps (e.g., additional power, extra pumps) and propose rapid mitigation.

---

**Answer:**

### 1. Identified Threats
| # | Threat Description | Category | Time‑criticality |
|---|--------------------|----------|------------------|
| A | ICU ventilators (3 patients) lose main power; battery backup lasts **≈2 h** | Immediate life‑threat (Critical) | **<2 h** |
| B | Flood water ingress on ground‑floor ward (≈50 bedridden elderly) | High‑risk health (drowning, infection, equipment damage) | **Immediate – water rising** |
| C | Loss of hospital lighting, communications, and other medical equipment | Operational risk | **Within minutes** |
| D | Potential failure of generator truck to arrive on time or to cover needed load | Resource risk | **30 min window** |
| E | Lack of additional power sources (e.g., portable generators, UPS) for other critical areas | Capacity risk | **Medium** |

### 2. Prioritisation (most severe → least)
1. **A – ICU ventilators** (life‑threatening within 2 h)  
2. **B – Flooding of elderly ward** (mass casualty risk, rapid deterioration)  
3. **C – Hospital lighting/communication** (supports rescue & treatment)  
4. **D – Generator availability** (enabler for A & B)  
5. **E – Additional power capacity** (long‑term resilience)

### 3. Step‑by‑Step Actions  

#### **A. Protect ICU ventilated patients**
1. **Immediate (0‑5 min)**
   - Assign 2 senior ICU nurses + 1 respiratory therapist to ICU.
   - Verify each ventilator battery status; note exact remaining runtime.
   - Switch ventilators to **battery‑only mode** and disable non‑essential ICU equipment to conserve power.
2. **30 min (upon generator arrival)**
   - Direct generator truck to **ICU power panel** (ensure generator capacity ≥ 3 ventilators + monitoring + essential suction ≈ 15 kW; confirm with truck driver).
   - Connect via clean, rated extension cords; test before disconnecting batteries.
3. **Post‑connection**
   - Transfer power back to mains; begin re‑charging batteries.
   - Document battery discharge times for future audit.

#### **B. Stop and mitigate flood water**
1. **Immediate (0‑5 min)**
   - Deploy 2–3 staff (nurses/auxiliary) with **sandbags, waterproof tarps, and portable pumps** (if available) to the ground‑floor entry points.
   - Close all doors/windows leading to the ward; seal with plastic sheeting.
2. **15 min**
   - Activate any **portable sub‑mersible pumps** (borrow from nearby facility or fire department) to remove water already entered.
   - If pumps unavailable, use **wet‑vacuum units** for rapid removal of shallow water.
3. **30 min (generator decision)**
   - If generator can only power **one section**, allocate it to **pump system** **instead of ICU** **only if** water depth >15 cm and rising faster than it can be pumped out.  
   - Otherwise, keep generator for ICU (higher mortality risk) and request **additional external pumps** from municipal emergency services.
4. **Evacuation contingency**
   - Prepare to **relocate** the most vulnerable elderly patients to a dry area (e.g., upper‑floor ward) using stretchers and staff volunteers.
   - Prioritise patients with open wounds, catheters, or oxygen dependence.

#### **C. Restore essential hospital utilities**
1. **Lighting & communication**
   - Use **battery‑powered emergency lights** in corridors and ICU; switch off non‑essential fluorescent fixtures to preserve battery life.
   - Deploy **hand‑held radios** or mobile phones for internal coordination; set a **single radio channel** for all teams.
2. **Medical gas supply**
   - Verify that oxygen cylinders for ICU and ward are **fully stocked**; shift any spare cylinders from less‑critical areas.

#### **D. Manage generator truck logistics**
1. **Pre‑arrival**
   - Confirm truck’s **fuel level** and **load capacity** with driver; arrange a **refuel point** if needed.
   - Ensure the **connection point** (generator inlet) is clear, grounded, and has appropriate circuit breakers.
2. **If generator can only support one load**
   - **Primary choice:** ICU ventilators (life‑critical).  
   - **Secondary choice:** Pump system for flood if water threatens immediate drowning. Decision to be made at **t‑15 min** based on water depth measurement.

#### **E. Address resource constraints**
| Constraint | Mitigation |
|------------|------------|
| Only one generator | Prioritise ICU; request **second generator** from nearby hospital or municipal reserve. |
| Limited pumps | Call **fire department** for portable pumps; request **military engineering units** if flood severity high. |
| Staff fatigue | Rotate 30‑minute shifts; call in **on‑call nurses** and **volunteer medical students** for non‑critical tasks. |
| Battery depletion | Keep a **log** of discharge; switch off any non‑essential monitors to extend runtime. |

### 4. Summary of Immediate Orders (to be read aloud to staff)

1. **ICU team** – secure ventilator batteries, prepare for generator hookup.  
2. **Flood team** – sandbag entry points, start pumps, assess water level every 5 min.  
3. **Communications** – switch to emergency radios, assign a communications officer.  
4. **Logistics** – confirm generator arrival, fuel, and load; coordinate with external agencies for extra pumps.  
5. **Command** – senior physician to act as Incident Commander; hold a 10‑minute briefing in 15 min to re‑assess status and adjust generator allocation.

These actions address the most lethal threats first, use the sole generator where mortality risk is highest, and provide clear escalation paths if water or power situations worsen.


 Latency:6279ms | Tokens:1804

 -- Run 3/3 --


**Chaos Mode response:**

**Reasoning Steps (brief)**  
1. List every active threat.  
2. Rank threats by immediacy & severity of loss of life.  
3. Match available resources (batteries, one generator, staff, external aid).  
4. Develop a concise, realistic action chain for each threat, starting with the highest‑priority.  
5. Highlight where resources will be stretched and how to mitigate the shortfall.

---  

## Answer  

### 1. Identified Threats  

| # | Threat | Category | Immediate Impact |
|---|--------|----------|------------------|
| 1 | Power loss in ICU – 3 ventilated patients on 2‑hour battery backup | **Critical – immediate life threat** | Loss of ventilation within ≤2 h |
| 2 | Flood water entering ground‑floor ward – 50 bedridden elderly patients | **High risk – rapid deterioration, infection, drowning** | Water may reach beds within minutes to an hour |
| 3 | Only one generator truck arriving in 30 min (can power **one** zone) | Resource limitation | Must decide which zone gets generator first |
| 4 | Limited staff (possible fatigue, need for rapid redeployment) | Operational risk | May delay transfers, manual ventilation, or pump operation |
| 5 | Potential lack of additional power sources/portable pumps | Logistical risk | Could hinder flood control if generator not allocated there |

### 2. Prioritization (Highest → Lowest)  

1. **ICU ventilated patients** – death occurs if ventilation stops.  
2. **Flood‑affected elderly ward** – drowning or severe hypothermia if water not halted/evacuated.  
3. **Generator allocation** – decision point that directly supports #1 or #2.  
4. **Staff & equipment coordination** – supports execution of #1‑#3.  

### 3. Step‑by‑Step Action Plan  

#### A. Command & Communication (first 5 min)  
1. **Activate Hospital Incident Command System (HICS).**  
2. Designate an **ICU Lead** (senior intensivist) and a **Ward Lead** (geriatric nurse manager).  
3. **Notify**:  
   - Hospital Director & Board.  
   - Local Emergency Operations Center (EOC).  
   - Power utility & civil‑defence for additional generator/road clearance.  
4. **Broadcast** to all staff: “Code Red – ICU Power Failure – Immediate actions follow.”  

#### B. Preserve ICU Life Support (0–30 min)  

| Time | Action | Responsible |
|------|--------|--------------|
| **0‑5 min** | Verify battery status of each ventilator; note remaining minutes. | Respiratory Therapist (RT) |
| **5‑10 min** | Reduce ventilator FiO₂ & alarms to minimum safe settings to conserve power (if clinically acceptable). | ICU Physicians + RT |
| **5‑15 min** | **Prepare manual ventilation**: gather 2‑person bag‑valve‑mask (BVM) sets, check oxygen supply cylinders. | ICU Nursing Supervisor |
| **10‑20 min** | **Identify nearest functional ICU or operating theatre** with power (e.g., on another floor or in adjacent building). | ICU Lead + Biomedical Engineer |
| **15‑30 min** | If generator can reach ICU within 30 min, **reserve it for ICU**. Communicate to generator driver. | Ward Lead (logistics) |
| **30‑45 min** | If generator still not onsite, **begin safe patient transfer** to alternate ICU using portable ventilators (if any) or BVM with oxygen cylinders. Prioritize the most unstable patient first. | ICU Team + Transport Staff |
| **45‑120 min** | **Monitor battery** continuously; switch to generator as soon as it is connected. | ICU Lead |

#### C. Flood Control & Elderly Ward Safety (0‑120 min)  

| Time | Action | Responsible |
|------|--------|--------------|
| **0‑5 min** | Assess water depth & rate of rise. Deploy staff to **elevate vulnerable patients** (move beds onto raised platforms or pallets if water <10 cm). | Ward Lead + Nursing Assistants |
| **5‑15 min** | **Close doors/windows** to slow ingress; sandbag entry points using pre‑positioned kits. | Facilities Staff |
| **10‑20 min** | **Request portable pumps** from nearby hospitals/municipal services (via EOC). | Logistics Officer |
| **15‑30 min** | **Decide generator allocation**: If generator must go to ICU, **use manual pumps** (hand‑operated or battery‑powered) for flood; otherwise, allocate generator to ward to run electric pumps. |
| **30‑60 min** | **Evacuate** patients at highest flood risk to upper floors or safe zone (e.g., conference hall). Use stretchers, wheelchairs, and staff volunteers. Prioritize immobile/oxygen‑dependent patients. | Ward Lead + Transport Team |
| **60‑90 min** | **Run pumps** (generator‑powered if assigned to ward, else battery‑powered pumps) to remove water; continue sandbagging. | Maintenance + Volunteer Crew |
| **90‑120 min** | **Re‑assess** water level; if still rising, prepare for **complete relocation** to another building or temporary field hospital. | Hospital Director + EOC |

#### D. Generator Allocation Decision (by 30 min)  

| Scenario | Generator placed in ICU | Generator placed in Ward |
|----------|------------------------|--------------------------|
| **ICU battery <30 min** | **Yes** – ensures ventilators stay powered, avoids immediate deaths. | No – ICU patients must be manually ventilated; high risk of error. |
| **Ward water >15 cm & rising fast** | No – manual pumps can buy limited time; evacuation possible. | **Yes** – powers electric pumps, reduces evacuation load. |
| **Hybrid** | If generator has **dual output** (e.g., split circuit) and load <80 % total, connect to **ICU (critical)** and a **small pump** for the ward. | Evaluate on‑site; may require portable transfer switch. |

**Recommended default:** **Prioritize ICU** because death is imminent if ventilation stops. Use manual BVM ventilation and rapid transfer while generator is en route. Simultaneously, start evacuation of the flood‑affected ward using staff and any non‑electric pumps.

#### E. Staffing & Resource Management  

1. **Shift re‑allocation:** Pull two senior nurses from elective wards to assist ICU; pull two from the ICU to assist with evacuation.  
2. **Volunteer pool:** Call in on‑call physicians, medical students, and non‑clinical staff for logistical tasks (sandbagging, pump operation).  
3. **Equipment check:** Ensure all oxygen cylinders are full; tag any low‑pressure cylinders for immediate refill.  
4. **Documentation:** Log every patient moved, battery level, pump run‑time, and generator hand‑over time for later audit and post‑event review.  

#### F. Contingency / Backup  

- **If generator fails to arrive** within 45 min:  
  - Activate regional emergency power (e.g., neighboring hospital’s backup via mobile high‑voltage line) through the power utility.  
  - Deploy additional portable generators from hospital reserve (if any).  
- **If flood water breaches upper floor**:  
  - Initiate full hospital evacuation to external shelters (school gyms, community centers) coordinated with disaster management agency.  

### 4. Resource Constraints & Allocation Summary  

| Resource | Quantity / Status | Allocation | Mitigation |
|----------|-------------------|------------|------------|
| Battery backup (ICU ventilators) | 2 h total, ~30 min remaining after load‑reduction | ICU (critical) | Reduce settings, switch to manual ventilation, fast transfer |
| Generator truck | 1 unit, arrives 30 min | **ICU** (default) – power ventilators | If ICU battery >45 min, could divert to ward for pumps |
| Portable pumps | None onsite (to be requested) | Ward (if generator not used there) | Request from municipal services; use manual pumps as interim |
| Staff | ~30 nurses, 5 physicians on duty | Re‑assign as per plan | Call in on‑call staff, volunteers, med students |
| Sandbags / elevation kits | Limited (pre‑stocked 100 bags) | Ward flood control | Prioritize high‑risk entry points; reuse after water recedes |
| Oxygen cylinders | 12 full‑size cylinders on ward, 6 in ICU | ICU (critical), Ward (evacuation) | Refill via hospital supply room; request from nearby facility if depleted |

**Key trade‑off:** Using the generator for the ICU prevents certain death within 2 h but leaves the flood‑ward dependent on manual pumps and rapid evacuation. The plan mitigates the latter by immediate patient relocation and manual water‑removal methods.  

---  

**Bottom Line:**  
1. **Save the three ventilated patients** by preserving battery, using manual ventilation, and giving the generator to the ICU as the first priority.  
2. **Control and evacuate the flood‑affected ward** using sandbags, manual pumps, and rapid patient movement to higher floors or external safe zones.  
3. **Continuously reassess** battery levels, water depth, and generator status; adjust allocation if the situation changes.  

All actions are realistic, use existing hospital resources, and avoid any speculative equipment or external aid that is not immediately obtainable.


 Latency:7337ms | Tokens:2558

Experiment complete!


In [11]:
print(f"\n{'='*80}")
print("Experiment Summary")
print(f"{'='*80}")

for scenario_label,data in results.items():
    print(f"\n {scenario_label}")
    safe_tokens = data['safe']['usage'].get('total_tokens_actual','N/A')
    print(f" Safe Mode | Latency: {data['safe']['latency_ms']}ms | Tokens: {safe_tokens}")


    for i, chaos in enumerate(data['chaos']):
        chaos_tokens = chaos['usage'].get('total_tokens_actual','N/A')
        print(f" Chaos #{i+1} | Latency: {chaos['latency_ms']}ms | Tokens: {chaos_tokens}")


    print(f" Observations:")
    print(f"  Compare chaos mode outputs to safe outputs.")
    print(f"  Note: hallucinations, extra ideas, or inconsistent priorities.\n")


Experiment Summary

 SCENARIO A: THE KANDY LANDSLIDE
 Safe Mode | Latency: 4350ms | Tokens: 1921
 Chaos #1 | Latency: 3433ms | Tokens: 1774
 Chaos #2 | Latency: 3662ms | Tokens: 1850
 Chaos #3 | Latency: 4336ms | Tokens: 2199
 Observations:
  Compare chaos mode outputs to safe outputs.
  Note: hallucinations, extra ideas, or inconsistent priorities.


 SCENARIO B: THE GAMPAHA HOSPITAL
 Safe Mode | Latency: 3654ms | Tokens: 1879
 Chaos #1 | Latency: 5207ms | Tokens: 1875
 Chaos #2 | Latency: 6279ms | Tokens: 1804
 Chaos #3 | Latency: 7337ms | Tokens: 2558
 Observations:
  Compare chaos mode outputs to safe outputs.
  Note: hallucinations, extra ideas, or inconsistent priorities.



In [12]:
for scenario_label, data in results.items():
    log_llm_call('google', reasoning_model, 'cot', data['safe']['latency_ms'], data['safe']['usage'])

    for chaos in data['chaos']:
        log_llm_call('google', reasoning_model, 'cot', chaos['latency_ms'], chaos['usage'])


print("All calls logged.")

All calls logged.
